# Featured Event — Candidate Parsing & Drill-Down Debug Notebook

Loads the cached Brave candidates and walks through each pipeline stage so you can see exactly what's being kept, drilled, or dropped — and why.

**Prereq:** run the pipeline at least once with `BRAVE_CACHE_DIR` defaulting on (or set explicitly) so `Featured Event/Code/brave_cache/*.json` exists. This notebook reads from that cache.

In [126]:
import os, sys, json
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

import importlib, aggregator_drilldown
importlib.reload(aggregator_drilldown)
from aggregator_drilldown import expand_listicle
import importlib, aggregator_drilldown; importlib.reload(aggregator_drilldown)

# Point Python at the project's shared code modules
REPO = Path.cwd().resolve()
while REPO.name and not (REPO / 'NewsletterCreation').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'NewsletterCreation' / 'Code'))

from aggregator_drilldown import is_aggregator_url, drill_down_candidate, find_primary_url, _hostname
from event_date_filter import filter_candidates_by_date, upcoming_friday

CACHE_DIR = REPO / 'Featured Event' / 'Code' / 'brave_cache'
print('Cache dir:', CACHE_DIR)
print('Cache files:', [p.name for p in CACHE_DIR.glob('*.json')])

Cache dir: /Users/couch2coders/Documents/GitHub/newsletters/Featured Event/Code/brave_cache
Cache files: ['East_Cobb_Connect_round1.json']


## 1. Pick a cached newsletter round to inspect

In [101]:
NEWSLETTER = 'East_Cobb_Connect'
ROUND      = 1

cache_path = CACHE_DIR / f'{NEWSLETTER}_round{ROUND}.json'
candidates = json.loads(cache_path.read_text(encoding='utf-8'))
print(f'Loaded {len(candidates)} candidates from {cache_path.name}\n')

# Plain-text printout — full title, full URL, no truncation.
for i, c in enumerate(candidates, 1):
    print(f"[{i:2}] host={_hostname(c.get('url',''))}")
    print(f"     title: {c.get('title','')}")
    print(f"     url:   {c.get('url','')}")
    print()

Loaded 30 candidates from East_Cobb_Connect_round1.json

[ 1] host=eastcobbnews.com
     title: Weekend Events: Taste of East Cobb festival; concerts; more - East Cobb News
     url:   https://eastcobbnews.com/weekend-events-taste-of-east-cobb-festival-concerts-more/

[ 2] host=cobbcountycourier.com
     title: Things to do this weekend in Cobb County: Friday May 15 to Sunday, May 17 - Cobb Courier
     url:   https://cobbcountycourier.com/2026/05/things-to-do-this-weekend-in-cobb-county-friday-may-15-to-sunday-may-17/

[ 3] host=mdjonline.com
     title: OUT AND ABOUT: 5 things to do this weekend in Cobb County — May 15 - 17 | Frontpage | mdjonline.com
     url:   https://www.mdjonline.com/news/frontpage/out-and-about-5-things-to-do-this-weekend-in-cobb-county-may-15-/article_200a61d3-c605-4e51-bb96-356d410d1f32.html

[ 4] host=cobbcounty.gov
     title: Cobb's Calendar Highlights - Upcoming Events in the County | Cobb County Georgia
     url:   https://www.cobbcounty.gov/news/cobbs-c

## 2. Stage 1 — Listicle detection

Listicle pages ("5 things to do", "weekend checklist") get dropped *before* drilling because reducing a multi-event roundup to one primary URL causes mismatches downstream.

In [104]:
LISTICLE_MARKERS = (
    'things to do', 'things-to-do', '5 things', '10 things',
    'weekend checklist', 'weekend events', 'weekend roundup',
    'weekend guide', 'your weekend', 'events this weekend',
    'events this week', 'events you absolutely need',
    'out and about', 'what to do this', "what's happening",
    'upcoming events', 'calendar of events', 'events calendar',
    'fun things to do', 'guide to events', 'things to do in',
)

def is_listicle(c):
    blob = f"{c.get('title','')} {c.get('url','')}".lower()
    matched = [m for m in LISTICLE_MARKERS if m in blob]
    return matched

# Plain-text printout — sidesteps every truncation issue (pandas, Jupyter
# CSS, ipywidgets, etc.). One block per candidate, full URL on its own line.
for i, c in enumerate(candidates, 1):
    url = c.get('url', '')
    title = c.get('title', '')
    host = _hostname(url)
    is_agg = is_aggregator_url(url)
    matched = is_listicle(c) if is_agg else []
    verdict = 'LISTICLE' if matched else ('AGGREGATOR' if is_agg else 'normal')
    print(f"[{i:2}] {verdict:<11} host={host}")
    print(f"     title: {title}")
    print(f"     url:   {url}")
    if matched:
        print(f"     markers: {', '.join(matched)}")
    print()

[ 1] LISTICLE    host=eastcobbnews.com
     title: Weekend Events: Taste of East Cobb festival; concerts; more - East Cobb News
     url:   https://eastcobbnews.com/weekend-events-taste-of-east-cobb-festival-concerts-more/
     markers: weekend events

[ 2] LISTICLE    host=cobbcountycourier.com
     title: Things to do this weekend in Cobb County: Friday May 15 to Sunday, May 17 - Cobb Courier
     url:   https://cobbcountycourier.com/2026/05/things-to-do-this-weekend-in-cobb-county-friday-may-15-to-sunday-may-17/
     markers: things to do, things-to-do

[ 3] LISTICLE    host=mdjonline.com
     title: OUT AND ABOUT: 5 things to do this weekend in Cobb County — May 15 - 17 | Frontpage | mdjonline.com
     url:   https://www.mdjonline.com/news/frontpage/out-and-about-5-things-to-do-this-weekend-in-cobb-county-may-15-/article_200a61d3-c605-4e51-bb96-356d410d1f32.html
     markers: things to do, things-to-do, 5 things, out and about

[ 4] normal      host=cobbcounty.gov
     title: Cobb'

## 3. Stage 2 — Drill down each surviving aggregator URL

For every aggregator URL that **isn't** a listicle, fetch the page, score every outbound anchor by (URL-slug overlap × 4) + (heading overlap × 4) + (anchor-text overlap × 3), and swap the candidate's URL to the highest-scoring primary if it clears the threshold (MIN_SCORE = 4).

In [ ]:
import importlib, aggregator_drilldown
importlib.reload(aggregator_drilldown)
from aggregator_drilldown import expand_listicle, is_aggregator_url

# Mirrors the pipeline: every aggregator URL gets expanded into N event
# candidates (listicle or single-event page — same treatment).
for c in candidates:
    url = c.get('url', '')
    if not is_aggregator_url(url):
        continue
    title = c.get('title', '')
    print(f"AGGREGATOR  {url}")
    print(f"  title: {title}")
    expanded = expand_listicle(url)
    if not expanded:
        print(f"  ↳ no event links found, keeping original URL")
        print()
        continue
    print(f"  ↳ extracted {len(expanded)} event candidates:")
    for i, ev in enumerate(expanded, 1):
        print(f"     [{i:2}] {ev['title']} → {ev['url']}")
    print()

## 4. Inspect drill scoring on a single URL

Edit `inspect_url` below to dig into why a particular drill picked the URL it did. This calls `find_primary_url` directly — same code path as the pipeline.

In [132]:
inspect_url   = 'https://www.ajc.com/accessatl/2026/05/the-ultimate-weekend-checklist-10-metro-atlanta-events-you-absolutely-need-to-check-out/'
inspect_title = '20th Annual Taste of East Cobb 2026'

primary = find_primary_url(inspect_url, title=inspect_title)
print('Drill result:', primary or '(no primary chosen)')

      ↳ drilled 'https://www.ajc.com/accessatl/2026/05/the-ultimate-weekend-c…' → 'https://www.awesomealpharetta.com/taste-of-alpharetta' (anchor='awesomealpharetta.com', heading='Taste of Alpharetta', score=10)
Drill result: https://www.awesomealpharetta.com/taste-of-alpharetta


## 4b. Expand a listicle into individual event candidates

Instead of dropping a listicle, scrape it and pull out every plausible event URL inside. Each becomes its own candidate (with title from the surrounding heading where possible).

In [138]:
from aggregator_drilldown import expand_listicle

listicle_url = 'https://www.mdjonline.com/news/frontpage/out-and-about-5-things-to-do-this-weekend-in-cobb-county-may-15-/article_200a61d3-c605-4e51-bb96-356d410d1f32.html'
events = expand_listicle(listicle_url)
print(f'Extracted {len(events)} event candidates from {listicle_url}\n')
for i, ev in enumerate(events, 1):
    print(f"[{i:2}] {ev['title']}")
    print(f"     url:    {ev['url']}")
    if ev.get('summary'):
        print(f"     anchor: {ev['summary']}")
    print()

Extracted 14 event candidates from https://www.mdjonline.com/news/frontpage/out-and-about-5-things-to-do-this-weekend-in-cobb-county-may-15-/article_200a61d3-c605-4e51-bb96-356d410d1f32.html

[ 1] Art of the Cocktail
     url:    http://mariettacobbartmuseum.org
     anchor: Madeline Beck, curator for the Marietta-Cobb Museum of Art, examines a wooden bowl piece at the museum’s “Georgia Wood Artists” juried exhibition. Proceeds from the upcoming Art of the Cocktail event benefit the museum. from the Marietta Daily Journal Madeline Beck, curator for the Marietta-Cobb Museum of Art, examines a wooden bowl piece at the museum’s “Georgia Wood Artists” juried exhibition. Proceeds from the upcoming Art of the Cocktail event benefit the museum. Madeline Beck, curator for the Marietta-Cobb Museum of Art, examines a wooden bowl piece at the museum’s “Georgia Wood Artists” juried exhibition. Proceeds from the upcoming Art of the Cocktail event benefit the museum. from the Marietta Daily Journal 

## 5. Stage 3 — Date filter

After listicle drop + drill, the date filter checks title + summary + article body + primary-text for a date on or after the upcoming-Friday floor. Anything purely about past dates gets removed.

In [ ]:
# Build the post-listicle, post-drill pool the same way the pipeline does
pool = []
for c in candidates:
    url = c.get('url', '')
    if is_aggregator_url(url):
        if is_listicle(c):
            continue
        c2 = dict(c)
        drill_down_candidate(c2)
        pool.append(c2)
    else:
        pool.append(c)

floor = upcoming_friday()
print('Date floor:', floor, '\n')

kept, past_urls = filter_candidates_by_date(
    pool, floor,
    text_keys=('title', 'summary', 'article_text', 'primary_text'),
)
kept_urls = {c['url'] for c in kept}

# Plain-text printout — full URLs, no truncation. KEPT items first.
def _emit(c, status):
    print(f"[{status}]  host={_hostname(c.get('url',''))}")
    print(f"  title: {c.get('title','')}")
    print(f"  url:   {c.get('url','')}")
    dates = c.get('dates_found') or []
    if dates:
        print(f"  dates: {', '.join(dates)}")
    print()

for c in pool:
    if c.get('url') in kept_urls:
        _emit(c, 'KEPT')
for c in pool:
    if c.get('url') not in kept_urls:
        _emit(c, 'DROPPED (past-only)')

## 6. Final pool (what Claude sees)

In [ ]:
print(f'Final pool: {len(kept)} candidates\n')
for i, c in enumerate(kept, 1):
    print(f'{i}. {c.get("title","?")[:80]}')
    print(f'   {c.get("url","")}')
    print()